# 🫀 Left Atrium Segmentation với DINOv2

## 0. 🖥️ Kiểm tra GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️ Đang ở chế độ CPU')

## 1. 📂 Mount Google Drive & Thiết lập môi trường

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Left_atrium_segmentation

import os, sys
PROJECT_PATH = '/content/drive/MyDrive/Left_atrium_segmentation'
sys.path.insert(0, PROJECT_PATH)
sys.path.insert(0, os.path.join(PROJECT_PATH, 'src'))
print('Thư mục làm việc hiện tại:', os.getcwd())
print('Các thư mục có sẵn:', os.listdir('.'))

## 2. 📦 Cài đặt thư viện cần thiết

In [ ]:
!pip install -q nibabel scikit-learn tqdm matplotlib scipy
print('✅ Cài đặt thư viện hoàn tất!')

## 3. ⚡ Tăng tốc: Giải nén dữ liệu 2D sang SSD nội bộ Colab (`/content/data`)
*(Cần chạy mỗi lần mở phiên Colab mới nếu muốn train)*

In [ ]:
import os
zip_path = '/content/drive/MyDrive/Left_atrium_segmentation/data/data_2d.zip'

if os.path.exists(zip_path):
    print('[*] Tìm thấy data_2d.zip! Đang giải nén siêu tốc sang /content/data...')
    !mkdir -p /content/data
    !unzip -q -o "{zip_path}" -d /content/data/
    print('✅ Giải nén hoàn tất chỉ trong 3 giây!')
elif not os.path.exists('/content/data/train_2d/images'):
    print('[*] Đang copy nhanh dữ liệu qua luồng tar...')
    !mkdir -p /content/data
    !tar -cf - -C /content/drive/MyDrive/Left_atrium_segmentation/data train_2d val_2d test_2d | tar -xf - -C /content/data/
    print('✅ Chuyển dữ liệu hoàn tất!')
else:
    print('✅ Dữ liệu 2D đã có sẵn trên SSD nội bộ (/content/data)!')

## 4. 🚀 HUẤN LUYỆN MÔ HÌNH

### 4.1. Mục 1 — Huấn luyện 2D Decoder với BCE + Dice Loss - ✅ ĐÃ LƯU CHECKPOINT
*Checkpoint: `best_decoder_bce_dice.pth` | Best Val Dice: 0.8343 | Test Dice: 0.7376*

In [ ]:
!python -u src/train_fast.py \
    --model vit_large \
    --data_root /content/data \
    --batch_size 64 \
    --epochs 35 \
    --lr 0.001 \
    --patience 7 \
    --loss bce_dice \
    --save_dir results

### 4.2. Mục 5 — Huấn luyện 2.5D Context `[z-1, z, z+1]` - ✅ ĐÃ LƯU CHECKPOINT
*Checkpoint: `best_decoder_bce_dice_2_5d.pth` | Best Val Dice: 0.7965 | Test Dice: 0.7102*

In [ ]:
!python -u src/train_fast.py \
    --model vit_large \
    --data_root /content/data \
    --batch_size 64 \
    --epochs 35 \
    --lr 0.001 \
    --patience 7 \
    --loss bce_dice \
    --mode_2_5d \
    --save_dir results

### 4.3. Mục 2 — LoRA Fine-tuning Encoder - ✅ ĐÃ LƯU CHECKPOINT
*Checkpoint: `best_decoder_vit_large_bce_dice_lora.pth` | Best Val Dice: 0.8494 | Test Dice: 0.7819*

In [ ]:
!python -u src/train_lora.py \
    --model vit_large \
    --data_root /content/data \
    --batch_size 32 \
    --epochs 30 \
    --lr_decoder 0.001 \
    --lr_lora 0.00001 \
    --patience 7 \
    --lora_blocks 2 \
    --lora_rank 4 \
    --save_dir results

### 4.4. Mục 3 — FPN Multi-scale Decoder (Cải thiện chi tiết đường viền)
**Vấn đề:** DINOv2 chia ảnh thành patch 14×14, decoder gốc chỉ dùng đặc trưng từ lớp cuối cùng → đường viền tâm nhĩ bị vuông vức, mất chi tiết cạnh.

**Giải pháp FPN:**
- Lấy đặc trưng từ **4 lớp Transformer cuối** (lớp 9, 10, 11, 12 của ViT-S)
- Lớp nông: giữ thông tin cạnh và texture cục bộ
- Lớp sâu: giữ ngữ cảnh tổng thể về vị trí tâm nhĩ
- **Top-down FPN merge**: ngữ cảnh sâu truyền về lớp nông → kết hợp cả hai
- Concat 4 feature maps (512 channels) → upsample → dự đoán mask mượt hơn

In [ ]:
# === MỤC 3: FPN MULTI-SCALE DECODER (~2-4 PHÚT) ===
# Kết quả lưu tại: results/best_decoder_vit_large_bce_dice_fpn.pth
!python -u src/train_fpn.py \
    --model vit_large \
    --data_root /content/data \
    --batch_size 32 \
    --epochs 35 \
    --lr 0.001 \
    --patience 7 \
    --n_levels 4 \
    --save_dir results

## 5. 📊 ĐÁNH GIÁ TRÊN TẬP TEST (EVALUATION LAB)

### 5.1. Đánh giá mô hình 2D — Mục 1 (BCE + Dice Loss)

In [ ]:
!python -u src/evaluate.py \
    --model vit_large \
    --checkpoint results/best_decoder_bce_dice.pth \
    --data_root /content/data \
    --batch_size 32 \
    --num_workers 2 \
    --max_vis 20

### 5.2. Đánh giá mô hình 2D + Hậu xử lý LCC — Mục 4

In [ ]:
!python -u src/evaluate.py \
    --model vit_large \
    --checkpoint results/best_decoder_bce_dice.pth \
    --data_root /content/data \
    --use_lcc \
    --max_vis 20

### 5.3. Đánh giá mô hình 2.5D Context 3D — Mục 5

In [ ]:
!python -u src/evaluate.py \
    --model vit_large \
    --checkpoint results/best_decoder_bce_dice_2_5d.pth \
    --data_root /content/data \
    --batch_size 32 \
    --num_workers 2 \
    --max_vis 20

### 5.4. Đánh giá mô hình LoRA Fine-tuning — Mục 2

In [ ]:
import os, torch, sys
sys.path.insert(0, 'src')

from model import DINOv2Segmenter
from lora import inject_lora_into_backbone
from dataset import get_dataloaders
from train import dice_score, iou_score
from evaluate import evaluate_model
import numpy as np, json

CHECKPOINT = 'results/best_decoder_vit_large_bce_dice_lora.pth'
DATA_ROOT  = '/content/data'
SAVE_DIR   = 'results'
VIS_DIR    = 'results/visualizations_lora'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
lora_blocks = ckpt.get('lora_blocks', 2)
lora_rank   = ckpt.get('lora_rank', 4)
lora_alpha  = ckpt.get('lora_alpha', 4.0)
print(f'LoRA config: blocks={lora_blocks}, rank={lora_rank}, alpha={lora_alpha}')

model = DINOv2Segmenter(model_name='vit_large').to(device)
inject_lora_into_backbone(model.encoder.backbone, num_last_blocks=lora_blocks,
                          rank=lora_rank, alpha=lora_alpha)
model = model.to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('✅ Đã load checkpoint LoRA!')

loaders = get_dataloaders(data_root=DATA_ROOT, batch_size=16, num_workers=0)
all_dice, all_iou = evaluate_model(model, loaders['test'], device, VIS_DIR, max_vis=20)

print('\n' + '='*60)
print('KẾT QUẢ [LoRA Fine-tuning - Mục 2]:')
print(f'  Dice Mean: {np.mean(all_dice):.4f} | Dice Max: {np.max(all_dice):.4f}')
print(f'  IoU  Mean: {np.mean(all_iou):.4f}  | IoU  Max: {np.max(all_iou):.4f}')
print('='*60)

results = {
    'checkpoint': CHECKPOINT, 'mode_2_5d': False, 'use_lcc': False,
    'lora_blocks': lora_blocks, 'lora_rank': lora_rank,
    'num_samples': len(all_dice),
    'dice_mean': float(np.mean(all_dice)), 'dice_max': float(np.max(all_dice)),
    'dice_std':  float(np.std(all_dice)),  'dice_min': float(np.min(all_dice)),
    'iou_mean':  float(np.mean(all_iou)),  'iou_max':  float(np.max(all_iou)),
    'iou_std':   float(np.std(all_iou)),   'iou_min':  float(np.min(all_iou)),
    'per_sample_dice': all_dice, 'per_sample_iou': all_iou,
}
out_path = os.path.join(SAVE_DIR, 'test_results_best_decoder_vit_large_bce_dice_lora.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'  Kết quả chi tiết: {out_path}')

### 5.5. Đánh giá mô hình FPN Multi-scale Decoder — Mục 3

In [ ]:
# === MỤC 3: ĐÁNH GIÁ MÔ HÌNH FPN ===
import os, torch, sys
sys.path.insert(0, 'src')

from model import DINOv2FPNSegmenter
from dataset import get_dataloaders
from train import dice_score, iou_score
from evaluate import evaluate_model
import numpy as np, json

CHECKPOINT = 'results/best_decoder_vit_large_bce_dice_fpn.pth'
DATA_ROOT  = '/content/data'
SAVE_DIR   = 'results'
VIS_DIR    = 'results/visualizations_fpn'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ckpt     = torch.load(CHECKPOINT, map_location=device, weights_only=False)
n_levels = ckpt.get('n_levels', 4)
print(f'FPN config: n_levels={n_levels}')

model = DINOv2FPNSegmenter(model_name='vit_large', n_levels=n_levels).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('✅ Đã load checkpoint FPN!')

loaders = get_dataloaders(data_root=DATA_ROOT, batch_size=16, num_workers=0)
all_dice, all_iou = evaluate_model(model, loaders['test'], device, VIS_DIR, max_vis=20)

print('\n' + '='*60)
print('KẾT QUẢ [FPN Multi-scale Decoder - Mục 3]:')
print(f'  Dice Mean: {np.mean(all_dice):.4f} | Dice Max: {np.max(all_dice):.4f}')
print(f'  IoU  Mean: {np.mean(all_iou):.4f}  | IoU  Max: {np.max(all_iou):.4f}')
print('='*60)

results = {
    'checkpoint': CHECKPOINT, 'mode_2_5d': False, 'use_lcc': False,
    'n_levels': n_levels,
    'num_samples': len(all_dice),
    'dice_mean': float(np.mean(all_dice)), 'dice_max': float(np.max(all_dice)),
    'dice_std':  float(np.std(all_dice)),  'dice_min': float(np.min(all_dice)),
    'iou_mean':  float(np.mean(all_iou)),  'iou_max':  float(np.max(all_iou)),
    'iou_std':   float(np.std(all_iou)),   'iou_min':  float(np.min(all_iou)),
    'per_sample_dice': all_dice, 'per_sample_iou': all_iou,
}
out_path = os.path.join(SAVE_DIR, 'test_results_best_decoder_vit_large_bce_dice_fpn.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'  Kết quả chi tiết: {out_path}')

### 5.6. 📋 Bảng tổng hợp & So sánh tất cả các phương pháp

In [ ]:
import os, json, glob
import pandas as pd

result_files = sorted(glob.glob('results/test_results_*.json'))
if result_files:
    rows = []
    for rf in result_files:
        with open(rf, 'r', encoding='utf-8') as f:
            data = json.load(f)
        name = os.path.basename(rf).replace('test_results_', '').replace('.json', '')
        row = {
            'Phương pháp': name,
            'Chế độ':   '2.5D' if data.get('mode_2_5d') else '2D',
            'LCC':      'Bật' if data.get('use_lcc') else 'Tắt',
            'LoRA':     f"b={data['lora_blocks']},r={data['lora_rank']}" if data.get('lora_blocks') else '—',
            'FPN':      f"levels={data['n_levels']}" if data.get('n_levels') else '—',
            'Dice Mean': f"{data.get('dice_mean', 0):.4f}",
            'Dice Max':  f"{data.get('dice_max', 0):.4f}",
            'IoU Mean':  f"{data.get('iou_mean', 0):.4f}",
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    display(df)
else:
    print('Chưa có file kết quả. Hãy chạy các ô đánh giá ở trên trước.')

## 6. 📈 XEM BIỂU ĐỒ TRAINING CURVES

In [ ]:
import json, os
import matplotlib.pyplot as plt

# Chọn file history: fpn → lora → mặc định
for fname in ['results/training_history_fpn.json',
              'results/training_history_lora.json',
              'results/training_history.json']:
    if os.path.exists(fname):
        history_file = fname
        break

if os.path.exists(history_file):
    with open(history_file) as f:
        history = json.load(f)

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, (metric, title) in zip(axes, [('loss','Loss'),('dice','Dice'),('iou','IoU')]):
        ax.plot(epochs, history[f'train_{metric}'], 'b-o', markersize=4, label='Train')
        ax.plot(epochs, history[f'val_{metric}'],   'r-o', markersize=4, label='Val')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Training Curves: {os.path.basename(history_file)}',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Chưa có file history.')

## 7. 🖼️ TRỰC QUAN HÓA HÌNH ẢNH KẾT QUẢ

In [ ]:
import glob, os
from IPython.display import Image, display

vis_dirs = ['results/visualizations_fpn', 'results/visualizations_lora',
            'results/visualizations', 'results/visualizations_lcc',
            'results/visualizations_2_5d']
for vd in vis_dirs:
    if os.path.exists(vd):
        vis_files = sorted(glob.glob(f'{vd}/*.png'))
        if vis_files:
            print(f'\n' + '='*60)
            print(f'=== HÌNH ẢNH TỪ: {vd} ===')
            print('='*60)
            for f in vis_files[:4]:
                print(f'📸 {os.path.basename(f)}')
                display(Image(f, width=800))
            break